# Conjugate Priors: Practical Examples for Commodity Markets

## Learning Objectives

By completing this notebook, you will:
1. Understand conjugate prior-likelihood pairs and why they're useful
2. Apply Beta-Binomial conjugacy to commodity trading success rates
3. Use Normal-Normal conjugacy for price level estimation
4. Apply Gamma-Poisson conjugacy to commodity delivery counts
5. Visualize how data updates prior beliefs analytically
6. Recognize when conjugacy breaks down and MCMC is needed

## Prerequisites
- Bayes' theorem understanding
- Familiarity with common probability distributions
- Basic calculus (for understanding derivations)

## Estimated Time: 75 minutes

---

In [ ]:
learning_objectives(["Understand conjugate prior-likelihood pairs and why they're useful", "Apply Beta-Binomial conjugacy to commodity trading success rates", "Use Normal-Normal conjugacy for price level estimation", "Apply Gamma-Poisson conjugacy to commodity delivery counts", "Visualize how data updates prior beliefs analytically", "Recognize when conjugacy breaks down and MCMC is needed"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

## 1. Setup and Imports

We'll use standard scientific Python libraries and visualization tools.

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configuration
np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Environment ready!")

## 2. What is Conjugacy?

**Conjugacy** is a special property where:
- The prior and posterior belong to the same family of distributions
- The posterior can be computed analytically (no MCMC needed)
- Updates are simply parameter transformations

**Mathematical Definition:**

Given:
- Prior: $p(\theta) = \text{Dist}(\theta; \alpha, \beta)$
- Likelihood: $p(\mathbf{x} | \theta) = \text{Likelihood}(\mathbf{x}; \theta)$

If the posterior is:
$$p(\theta | \mathbf{x}) = \text{Dist}(\theta; \alpha', \beta')$$

where $\alpha', \beta'$ are updated parameters, then the prior is **conjugate** to the likelihood.

**Why it matters:**
1. **Speed**: No sampling required, instant inference
2. **Intuition**: Parameter updates have clear interpretations
3. **Online learning**: Easy sequential updates as new data arrives
4. **Foundation**: Understanding conjugacy builds intuition for MCMC

## 3. Example 1: Beta-Binomial for Trading Success Rate

**Scenario:** A commodity trader has a track record of profitable vs unprofitable trades. We want to estimate their true success rate $\theta$.

**Model:**
- Prior: $\theta \sim \text{Beta}(\alpha, \beta)$
- Likelihood: $k | \theta \sim \text{Binomial}(n, \theta)$ where $k$ = successes out of $n$ trades
- Posterior: $\theta | k \sim \text{Beta}(\alpha + k, \beta + n - k)$

**Interpretation:**
- $\alpha$: Prior "pseudo-successes"
- $\beta$: Prior "pseudo-failures"
- Data adds $k$ successes and $n-k$ failures

In [ ]:
# Trading scenario
# Prior belief: Trader is probably mediocre (50% success rate)
# Beta(5, 5) has mean = 5/(5+5) = 0.5, but allows for variation

alpha_prior = 5
beta_prior = 5

# Observed data: 15 successful trades out of 20
n_trades = 20
n_successes = 15
n_failures = n_trades - n_successes

# Posterior parameters (conjugate update)
alpha_post = alpha_prior + n_successes
beta_post = beta_prior + n_failures

print(f"Prior: Beta({alpha_prior}, {beta_prior})")
print(f"Data: {n_successes} successes out of {n_trades} trades")
print(f"Posterior: Beta({alpha_post}, {beta_post})")
print(f"\nPrior mean: {alpha_prior/(alpha_prior + beta_prior):.3f}")
print(f"Posterior mean: {alpha_post/(alpha_post + beta_post):.3f}")
print(f"Sample proportion: {n_successes/n_trades:.3f}")

In [ ]:
# Visualize prior, likelihood, and posterior
theta_grid = np.linspace(0, 1, 1000)

# Prior distribution
prior_dist = stats.beta(alpha_prior, beta_prior)
prior_pdf = prior_dist.pdf(theta_grid)

# Likelihood (proportional to binomial probability as function of theta)
likelihood = stats.binom(n_trades, theta_grid).pmf(n_successes)
# Normalize for plotting
likelihood = likelihood / np.trapz(likelihood, theta_grid)

# Posterior distribution
post_dist = stats.beta(alpha_post, beta_post)
post_pdf = post_dist.pdf(theta_grid)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(theta_grid, prior_pdf, 'b-', linewidth=2, label='Prior', alpha=0.7)
ax.plot(theta_grid, likelihood, 'g--', linewidth=2, label='Likelihood', alpha=0.7)
ax.plot(theta_grid, post_pdf, 'r-', linewidth=3, label='Posterior', alpha=0.9)

# Mark means
ax.axvline(alpha_prior/(alpha_prior + beta_prior), color='blue', 
           linestyle=':', alpha=0.5, label='Prior Mean')
ax.axvline(alpha_post/(alpha_post + beta_post), color='red', 
           linestyle=':', alpha=0.5, label='Posterior Mean')
ax.axvline(n_successes/n_trades, color='green', 
           linestyle=':', alpha=0.5, label='MLE')

ax.set_xlabel('Success Rate θ', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Beta-Binomial Conjugacy: Trading Success Rate Estimation', fontsize=14)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key Observations:**

1. **Posterior is a compromise** between prior (centered at 0.5) and data (15/20 = 0.75)
2. **Prior has influence**: With only 20 trades, the prior keeps the estimate from being too extreme
3. **Uncertainty reduction**: Posterior is narrower than prior (more confidence)
4. **Conjugacy makes this instant**: No sampling, just parameter arithmetic

### Sequential Learning: How Beliefs Update with More Data

One powerful feature of conjugate priors: we can update sequentially as new data arrives.

In [ ]:
# Simulate sequential trade results
np.random.seed(42)
true_theta = 0.70  # True (unknown) success rate
total_trades = 100

# Start with prior
alpha_current = 5
beta_current = 5

# Track posterior means over time
means = [alpha_current / (alpha_current + beta_current)]
lower_95 = [stats.beta(alpha_current, beta_current).ppf(0.025)]
upper_95 = [stats.beta(alpha_current, beta_current).ppf(0.975)]

# Update sequentially
for i in range(total_trades):
    # Simulate one trade
    success = np.random.rand() < true_theta
    
    # Update posterior
    if success:
        alpha_current += 1
    else:
        beta_current += 1
    
    # Record
    means.append(alpha_current / (alpha_current + beta_current))
    lower_95.append(stats.beta(alpha_current, beta_current).ppf(0.025))
    upper_95.append(stats.beta(alpha_current, beta_current).ppf(0.975))

# Plot evolution
fig, ax = plt.subplots(figsize=(12, 6))

trades = np.arange(len(means))
ax.plot(trades, means, 'b-', linewidth=2, label='Posterior Mean')
ax.fill_between(trades, lower_95, upper_95, alpha=0.3, label='95% Credible Interval')
ax.axhline(true_theta, color='red', linestyle='--', linewidth=2, 
           label=f'True Success Rate: {true_theta}')

ax.set_xlabel('Number of Trades', fontsize=12)
ax.set_ylabel('Estimated Success Rate', fontsize=12)
ax.set_title('Sequential Bayesian Learning: Beliefs Update with Each Trade', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final estimate: {means[-1]:.3f} (True: {true_theta})")
print(f"Final 95% CI: [{lower_95[-1]:.3f}, {upper_95[-1]:.3f}]")

## 4. Example 2: Normal-Normal for Oil Price Level

**Scenario:** Estimate the mean price of WTI crude oil based on recent observations.

**Model (known variance case):**
- Prior: $\mu \sim \mathcal{N}(\mu_0, \sigma_0^2)$
- Likelihood: $x_i | \mu \sim \mathcal{N}(\mu, \sigma^2)$ (assume $\sigma$ known)
- Posterior: $\mu | \mathbf{x} \sim \mathcal{N}(\mu_n, \sigma_n^2)$

where:
$$\mu_n = \frac{\sigma^2 \mu_0 + n \sigma_0^2 \bar{x}}{\sigma^2 + n \sigma_0^2}$$
$$\sigma_n^2 = \frac{\sigma^2 \sigma_0^2}{\sigma^2 + n \sigma_0^2}$$

**Interpretation:**
- Posterior mean is precision-weighted average of prior mean and data mean
- Posterior variance decreases with more data (precision adds)

In [ ]:
# Oil price estimation
# Prior: Historical average around $75, but uncertain (std = $15)
mu_0 = 75.0  # Prior mean
sigma_0 = 15.0  # Prior std

# Data: Recent weekly observations
# Assume we know daily volatility is about $5
sigma_data = 5.0  # Known data std

# Observed prices (simulated)
np.random.seed(42)
true_mu = 82.0  # True mean (unknown)
n_obs = 10
observed_prices = np.random.normal(true_mu, sigma_data, n_obs)
x_bar = observed_prices.mean()

print(f"Prior: N({mu_0}, {sigma_0**2})")
print(f"Observed data: {n_obs} prices, mean = ${x_bar:.2f}")
print(f"Assumed data std: ${sigma_data}")

In [ ]:
# Compute posterior parameters analytically
precision_prior = 1 / sigma_0**2
precision_data = n_obs / sigma_data**2
precision_post = precision_prior + precision_data

mu_post = (precision_prior * mu_0 + precision_data * x_bar) / precision_post
sigma_post = np.sqrt(1 / precision_post)

print(f"\nPosterior: N({mu_post:.2f}, {sigma_post**2:.2f})")
print(f"\nPosterior mean: ${mu_post:.2f}")
print(f"Posterior std: ${sigma_post:.2f}")
print(f"\n95% Credible Interval: [${mu_post - 1.96*sigma_post:.2f}, ${mu_post + 1.96*sigma_post:.2f}]")

In [ ]:
# Visualize
mu_grid = np.linspace(50, 100, 1000)

prior_pdf = stats.norm(mu_0, sigma_0).pdf(mu_grid)
likelihood_pdf = stats.norm(x_bar, sigma_data / np.sqrt(n_obs)).pdf(mu_grid)
post_pdf = stats.norm(mu_post, sigma_post).pdf(mu_grid)

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(mu_grid, prior_pdf, 'b-', linewidth=2, label=f'Prior: N({mu_0}, {sigma_0:.1f}²)', alpha=0.7)
ax.plot(mu_grid, likelihood_pdf, 'g--', linewidth=2, 
        label=f'Likelihood (scaled): N({x_bar:.1f}, {(sigma_data/np.sqrt(n_obs)):.1f}²)', alpha=0.7)
ax.plot(mu_grid, post_pdf, 'r-', linewidth=3, 
        label=f'Posterior: N({mu_post:.1f}, {sigma_post:.1f}²)', alpha=0.9)

# Mark true value
ax.axvline(true_mu, color='black', linestyle=':', linewidth=2, label=f'True μ: ${true_mu}')

ax.set_xlabel('Mean Oil Price μ ($)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Normal-Normal Conjugacy: Oil Price Level Estimation', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Precision Interpretation:**

Precision = 1/variance is the "information content". The posterior precision is the **sum** of prior and data precisions:

$$\text{Precision}_{\text{post}} = \text{Precision}_{\text{prior}} + \text{Precision}_{\text{data}}$$

This is why more data $\rightarrow$ higher precision $\rightarrow$ lower uncertainty.

## 5. Example 3: Gamma-Poisson for Delivery Counts

**Scenario:** A commodity warehouse receives delivery trucks. Estimate the average rate $\lambda$ (trucks per day).

**Model:**
- Prior: $\lambda \sim \text{Gamma}(\alpha, \beta)$
- Likelihood: $x_i | \lambda \sim \text{Poisson}(\lambda)$
- Posterior: $\lambda | \mathbf{x} \sim \text{Gamma}(\alpha + \sum x_i, \beta + n)$

**Interpretation:**
- $\alpha$: Prior "pseudo-count" of events
- $\beta$: Prior "pseudo-time" units
- Data adds total count and number of observations

In [ ]:
# Delivery truck scenario
# Prior: Expect around 5 trucks/day, with moderate uncertainty
# Gamma(25, 5) has mean = 25/5 = 5, std = sqrt(25)/5 = 1

alpha_prior = 25
beta_prior = 5

# Observed data: Daily truck counts over 10 days
np.random.seed(42)
true_lambda = 6.5  # True rate (unknown)
observed_counts = np.random.poisson(true_lambda, 10)

n_days = len(observed_counts)
total_count = observed_counts.sum()

# Posterior parameters
alpha_post = alpha_prior + total_count
beta_post = beta_prior + n_days

print(f"Prior: Gamma({alpha_prior}, {beta_prior})")
print(f"Prior mean: {alpha_prior/beta_prior:.2f} trucks/day")
print(f"\nObserved counts: {observed_counts}")
print(f"Total trucks: {total_count} over {n_days} days")
print(f"Sample mean: {total_count/n_days:.2f} trucks/day")
print(f"\nPosterior: Gamma({alpha_post}, {beta_post})")
print(f"Posterior mean: {alpha_post/beta_post:.2f} trucks/day")
print(f"True rate: {true_lambda} trucks/day")

In [ ]:
# Visualize
lambda_grid = np.linspace(0, 15, 1000)

prior_pdf = stats.gamma(alpha_prior, scale=1/beta_prior).pdf(lambda_grid)
post_pdf = stats.gamma(alpha_post, scale=1/beta_post).pdf(lambda_grid)

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(lambda_grid, prior_pdf, 'b-', linewidth=2, label='Prior', alpha=0.7)
ax.plot(lambda_grid, post_pdf, 'r-', linewidth=3, label='Posterior', alpha=0.9)
ax.axvline(true_lambda, color='black', linestyle=':', linewidth=2, 
           label=f'True Rate: {true_lambda}')
ax.axvline(alpha_post/beta_post, color='red', linestyle='--', linewidth=2, 
           label=f'Posterior Mean: {alpha_post/beta_post:.2f}')

ax.set_xlabel('Rate λ (trucks/day)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Gamma-Poisson Conjugacy: Delivery Rate Estimation', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. When Conjugacy Breaks Down

Conjugate priors are **convenient but restrictive**. They work only for specific likelihood families.

**Conjugacy doesn't work when:**
1. Likelihood is non-standard (e.g., custom commodity pricing model)
2. Multiple parameters with complex dependencies
3. Hierarchical structures
4. Need flexible priors that don't match conjugate families

**Example where we need MCMC:** Nonlinear regression

$$y_i = \alpha + \beta \exp(\gamma x_i) + \epsilon_i$$

No conjugate prior exists for $\gamma$ in this model. We need MCMC (which we'll cover in Module 6).

---

## Exercise 1: Beta-Binomial with Different Priors

**Task:** A new trader has 8 successes out of 10 trades. Compare posteriors under three different priors:

1. **Skeptical**: Beta(2, 8) — expects mostly failures
2. **Neutral**: Beta(1, 1) — uniform, no opinion
3. **Optimistic**: Beta(8, 2) — expects mostly successes

Plot all three posteriors on one graph and compute posterior means.

**Question:** Which prior gets "overwhelmed" by the data least/most?

In [ ]:
# YOUR CODE HERE
# Define priors, compute posteriors, and plot

n_trades_ex1 = 10
n_successes_ex1 = 8

# Define three priors
priors = {
    'Skeptical': (2, 8),
    'Neutral': (1, 1),
    'Optimistic': (8, 2)
}

# Compute posteriors
posteriors = {}
for name, (alpha, beta) in priors.items():
    # YOUR CODE: Compute posterior parameters
    alpha_post_ex1 = None  # <- Fill this in
    beta_post_ex1 = None   # <- Fill this in
    posteriors[name] = (alpha_post_ex1, beta_post_ex1)

# Plot
theta_grid_ex1 = np.linspace(0, 1, 1000)
fig, ax = plt.subplots(figsize=(12, 6))

for name, (alpha_post_ex1, beta_post_ex1) in posteriors.items():
    if alpha_post_ex1 is not None:
        post_pdf_ex1 = stats.beta(alpha_post_ex1, beta_post_ex1).pdf(theta_grid_ex1)
        ax.plot(theta_grid_ex1, post_pdf_ex1, linewidth=2, label=f'{name} Posterior')

ax.axvline(n_successes_ex1 / n_trades_ex1, color='black', linestyle='--', 
           label='Data proportion (MLE)')
ax.set_xlabel('Success Rate θ')
ax.set_ylabel('Density')
ax.set_title('Effect of Different Priors on Posterior (8/10 successes)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

# Print means
for name, (alpha_post_ex1, beta_post_ex1) in posteriors.items():
    if alpha_post_ex1 is not None:
        mean_ex1 = alpha_post_ex1 / (alpha_post_ex1 + beta_post_ex1)
        print(f"{name} posterior mean: {mean_ex1:.3f}")

In [ ]:
# AUTO-GRADED TEST
def test_exercise_1():
    """Verify posterior calculations."""
    # Check that posteriors were computed
    for name, (alpha, beta) in posteriors.items():
        assert alpha is not None, f"{name} posterior alpha is None"
        assert beta is not None, f"{name} posterior beta is None"
        
        # Verify correct update
        prior_alpha, prior_beta = priors[name]
        expected_alpha = prior_alpha + n_successes_ex1
        expected_beta = prior_beta + (n_trades_ex1 - n_successes_ex1)
        
        assert alpha == expected_alpha, (
            f"{name}: Expected alpha={expected_alpha}, got {alpha}. "
            "Remember: alpha_post = alpha_prior + successes"
        )
        assert beta == expected_beta, (
            f"{name}: Expected beta={expected_beta}, got {beta}. "
            "Remember: beta_post = beta_prior + failures"
        )
    
    print("✅ Exercise 1 passed!")

test_exercise_1()

---

## Exercise 2: Normal-Normal with Varying Data Size

**Task:** Show how the posterior depends on sample size. Use:
- Prior: N(70, 10²)
- True mean: 80
- Data std: 8
- Sample sizes: n = 1, 5, 10, 50, 100

For each sample size:
1. Generate n observations from N(80, 8²)
2. Compute posterior
3. Plot how posterior mean and std change with n

In [ ]:
# YOUR CODE HERE
np.random.seed(42)

mu_0_ex2 = 70
sigma_0_ex2 = 10
true_mu_ex2 = 80
sigma_data_ex2 = 8
sample_sizes = [1, 5, 10, 50, 100]

posterior_means = []
posterior_stds = []

for n in sample_sizes:
    # Generate data
    data_ex2 = np.random.normal(true_mu_ex2, sigma_data_ex2, n)
    x_bar_ex2 = data_ex2.mean()
    
    # YOUR CODE: Compute posterior parameters
    precision_prior_ex2 = None  # <- Fill this in
    precision_data_ex2 = None   # <- Fill this in
    precision_post_ex2 = None   # <- Fill this in
    
    mu_post_ex2 = None          # <- Fill this in
    sigma_post_ex2 = None       # <- Fill this in
    
    posterior_means.append(mu_post_ex2)
    posterior_stds.append(sigma_post_ex2)

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Posterior mean vs sample size
axes[0].plot(sample_sizes, posterior_means, 'o-', linewidth=2, markersize=8)
axes[0].axhline(true_mu_ex2, color='red', linestyle='--', label='True Mean')
axes[0].axhline(mu_0_ex2, color='blue', linestyle=':', label='Prior Mean')
axes[0].set_xlabel('Sample Size n')
axes[0].set_ylabel('Posterior Mean')
axes[0].set_title('Posterior Mean Converges to True Value')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Posterior std vs sample size
axes[1].plot(sample_sizes, posterior_stds, 'o-', color='purple', linewidth=2, markersize=8)
axes[1].axhline(sigma_0_ex2, color='blue', linestyle=':', label='Prior Std')
axes[1].set_xlabel('Sample Size n')
axes[1].set_ylabel('Posterior Std')
axes[1].set_title('Uncertainty Decreases with More Data')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# AUTO-GRADED TEST
def test_exercise_2():
    """Verify posterior calculations."""
    # Check that computations were done
    assert all(m is not None for m in posterior_means), "Some posterior means are None"
    assert all(s is not None for s in posterior_stds), "Some posterior stds are None"
    
    # Check that uncertainty decreases
    assert posterior_stds[0] > posterior_stds[-1], (
        f"Posterior std should decrease with sample size. "
        f"Got: n=1 std={posterior_stds[0]:.2f}, n=100 std={posterior_stds[-1]:.2f}"
    )
    
    # Check that posterior mean moves toward data
    # With large n, posterior should be closer to true_mu than prior
    final_mean = posterior_means[-1]
    assert abs(final_mean - true_mu_ex2) < abs(mu_0_ex2 - true_mu_ex2), (
        f"With n=100, posterior mean should be closer to true value than prior. "
        f"Prior: {mu_0_ex2}, Posterior: {final_mean:.2f}, True: {true_mu_ex2}"
    )
    
    print("✅ Exercise 2 passed!")

test_exercise_2()

---

## Exercise 3: Gamma-Poisson Prior Selection

**Task:** You're told a warehouse receives "around 10 trucks per day, give or take 3 trucks."

1. Choose Gamma(α, β) prior parameters to match this information
   - Hint: Gamma mean = α/β, Gamma std = √α/β
2. Simulate 30 days of data from Poisson(10)
3. Compute and plot the posterior
4. Report the posterior mean and 95% credible interval

In [ ]:
# YOUR CODE HERE
np.random.seed(42)

# Step 1: Choose prior parameters
# Target: mean = 10, std = 3
# Gamma(α, β): mean = α/β, std = √α/β
# From mean = α/β and std = √α/β, we get α = mean²/std²

target_mean = 10
target_std = 3

# YOUR CODE: Calculate alpha and beta
alpha_prior_ex3 = None  # <- Fill this in
beta_prior_ex3 = None   # <- Fill this in

print(f"Prior: Gamma({alpha_prior_ex3:.1f}, {beta_prior_ex3:.2f})")
if alpha_prior_ex3 is not None and beta_prior_ex3 is not None:
    print(f"Prior mean: {alpha_prior_ex3/beta_prior_ex3:.1f}")
    print(f"Prior std: {np.sqrt(alpha_prior_ex3)/beta_prior_ex3:.1f}")

# Step 2: Simulate data
true_lambda_ex3 = 10
n_days_ex3 = 30
data_ex3 = np.random.poisson(true_lambda_ex3, n_days_ex3)

# Step 3: Compute posterior
# YOUR CODE: Calculate posterior parameters
alpha_post_ex3 = None  # <- Fill this in
beta_post_ex3 = None   # <- Fill this in

# Step 4: Report results
if alpha_post_ex3 is not None and beta_post_ex3 is not None:
    post_mean_ex3 = alpha_post_ex3 / beta_post_ex3
    post_dist_ex3 = stats.gamma(alpha_post_ex3, scale=1/beta_post_ex3)
    ci_lower_ex3 = post_dist_ex3.ppf(0.025)
    ci_upper_ex3 = post_dist_ex3.ppf(0.975)
    
    print(f"\nPosterior mean: {post_mean_ex3:.2f} trucks/day")
    print(f"95% Credible Interval: [{ci_lower_ex3:.2f}, {ci_upper_ex3:.2f}]")
    
    # Plot
    lambda_grid_ex3 = np.linspace(0, 20, 1000)
    prior_pdf_ex3 = stats.gamma(alpha_prior_ex3, scale=1/beta_prior_ex3).pdf(lambda_grid_ex3)
    post_pdf_ex3 = post_dist_ex3.pdf(lambda_grid_ex3)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(lambda_grid_ex3, prior_pdf_ex3, 'b-', linewidth=2, label='Prior', alpha=0.7)
    ax.plot(lambda_grid_ex3, post_pdf_ex3, 'r-', linewidth=3, label='Posterior', alpha=0.9)
    ax.axvline(true_lambda_ex3, color='black', linestyle='--', linewidth=2, label='True Rate')
    ax.set_xlabel('Rate λ (trucks/day)')
    ax.set_ylabel('Density')
    ax.set_title('Exercise 3: Gamma-Poisson Inference')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# AUTO-GRADED TEST
def test_exercise_3():
    """Verify prior and posterior calculations."""
    # Check prior parameters exist
    assert alpha_prior_ex3 is not None, "alpha_prior_ex3 is None"
    assert beta_prior_ex3 is not None, "beta_prior_ex3 is None"
    
    # Check prior matches target
    prior_mean = alpha_prior_ex3 / beta_prior_ex3
    prior_std = np.sqrt(alpha_prior_ex3) / beta_prior_ex3
    
    assert abs(prior_mean - target_mean) < 0.1, (
        f"Prior mean should be close to {target_mean}, got {prior_mean:.2f}. "
        "Check your formula for alpha and beta."
    )
    assert abs(prior_std - target_std) < 0.1, (
        f"Prior std should be close to {target_std}, got {prior_std:.2f}. "
        "Check your formula for alpha and beta."
    )
    
    # Check posterior parameters
    assert alpha_post_ex3 is not None, "alpha_post_ex3 is None"
    assert beta_post_ex3 is not None, "beta_post_ex3 is None"
    
    expected_alpha = alpha_prior_ex3 + data_ex3.sum()
    expected_beta = beta_prior_ex3 + n_days_ex3
    
    assert alpha_post_ex3 == expected_alpha, (
        f"Posterior alpha should be {expected_alpha}, got {alpha_post_ex3}. "
        "Remember: alpha_post = alpha_prior + sum(data)"
    )
    assert abs(beta_post_ex3 - expected_beta) < 0.01, (
        f"Posterior beta should be {expected_beta}, got {beta_post_ex3}. "
        "Remember: beta_post = beta_prior + n"
    )
    
    print("✅ Exercise 3 passed!")

test_exercise_3()

---

## Exercise 4: Interpretive Questions

Answer these conceptual questions:

**Q1:** Why is conjugacy computationally convenient?

*YOUR ANSWER:*

---

**Q2:** In the Beta-Binomial example, what happens to the posterior as n → ∞?

*YOUR ANSWER:*

---

**Q3:** For Normal-Normal conjugacy, precision (1/variance) is additive. Why is this a natural way to think about information?

*YOUR ANSWER:*

---

**Q4:** Give an example of a commodity forecasting problem where conjugacy would NOT work.

*YOUR ANSWER:*

---

## Solutions

### Exercise 1 Solution

In [ ]:
# SOLUTION
# posteriors = {}
# for name, (alpha, beta) in priors.items():
#     alpha_post_ex1 = alpha + n_successes_ex1
#     beta_post_ex1 = beta + (n_trades_ex1 - n_successes_ex1)
#     posteriors[name] = (alpha_post_ex1, beta_post_ex1)
#
# Result:
# - Skeptical: Beta(10, 10) → mean = 0.50
# - Neutral: Beta(9, 3) → mean = 0.75
# - Optimistic: Beta(16, 4) → mean = 0.80
#
# The neutral prior (uniform) gets "overwhelmed" most by data (closer to MLE = 0.80)
# The skeptical prior resists the most (stays furthest from data)

### Exercise 2 Solution

In [ ]:
# SOLUTION
# for n in sample_sizes:
#     data_ex2 = np.random.normal(true_mu_ex2, sigma_data_ex2, n)
#     x_bar_ex2 = data_ex2.mean()
#     
#     precision_prior_ex2 = 1 / sigma_0_ex2**2
#     precision_data_ex2 = n / sigma_data_ex2**2
#     precision_post_ex2 = precision_prior_ex2 + precision_data_ex2
#     
#     mu_post_ex2 = (precision_prior_ex2 * mu_0_ex2 + precision_data_ex2 * x_bar_ex2) / precision_post_ex2
#     sigma_post_ex2 = np.sqrt(1 / precision_post_ex2)
#     
#     posterior_means.append(mu_post_ex2)
#     posterior_stds.append(sigma_post_ex2)
#
# Key insight: As n increases, data precision dominates, pulling posterior toward sample mean

### Exercise 3 Solution

In [ ]:
# SOLUTION
# From Gamma(α, β): mean = α/β, variance = α/β²
# Therefore: α = mean²/variance = 10²/3² = 11.11
# And: β = mean/variance = 10/9 = 1.11
#
# alpha_prior_ex3 = target_mean**2 / target_std**2
# beta_prior_ex3 = target_mean / target_std**2
#
# alpha_post_ex3 = alpha_prior_ex3 + data_ex3.sum()
# beta_post_ex3 = beta_prior_ex3 + n_days_ex3

### Exercise 4 Solutions

**Q1:** Conjugacy allows closed-form posterior calculation (no sampling), making inference instant and exact.

**Q2:** As n → ∞, the data dominates the prior, and the posterior converges to a point mass at the true parameter (data overwhelms prior).

**Q3:** Precision represents "information content" - more precise measurements contain more information. Adding independent information sources (prior + data) naturally sums their precisions.

**Q4:** Any nonlinear model (e.g., exponential decay in storage theory, regime-switching models, or stochastic volatility) breaks conjugacy and requires MCMC.

---

## Summary

### Key Takeaways

1. **Conjugate priors** enable closed-form Bayesian inference for specific likelihood families
2. **Beta-Binomial**: Natural for proportions and success rates
3. **Normal-Normal**: Works for estimating means (when variance is known)
4. **Gamma-Poisson**: Ideal for count data and rates
5. **Sequential learning**: Conjugacy makes online updating trivial (yesterday's posterior = today's prior)
6. **Limitations**: Conjugacy requires specific model structures; complex models need MCMC

### What's Next

Next notebook: **Prior Sensitivity Analysis** - How robust are our conclusions to prior choices?

### Additional Resources

- Fink (1997): "A Compendium of Conjugate Priors" (comprehensive reference)
- Murphy (2007): "Conjugate Bayesian analysis of the Gaussian distribution"
- Gelman BDA3, Chapter 2: "Single-parameter models"

---

*Remember: Conjugacy is elegant but limiting. Most real-world models require MCMC. But understanding conjugacy builds the intuition for thinking about how data updates beliefs!*